# Inspeção inicial de um snapshot do CNPJ

Este notebook prepara um snapshot mensal a partir do ZIP original e valida se a extração ficou pronta para uso no POC.

## Fluxo recomendado no Colab

1. Fazer upload apenas do arquivo mensal original, por exemplo `2026-04.zip`.
2. Colocar esse arquivo em `/content/uploads/`.
3. Executar este notebook para montar a estrutura de trabalho em `/content/dados_brutos/cnpj/AAAA-MM/`.
4. Validar famílias, amostras e metadados.

Este notebook não depende de `01_subarquivos_zip` persistente. A camada intermediária é criada de forma temporária e removida ao final da preparação.

In [ ]:
from collections import defaultdict
from pathlib import Path
import json
import shutil
import zipfile

UPLOAD_DIR = Path('/content/uploads')
CAMINHO_ZIP_PRINCIPAL = UPLOAD_DIR / '2026-04.zip'
RAIZ_DADOS = Path('/content/dados_brutos/cnpj')
SNAPSHOT_MES = '2026-04'

UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
CAMINHO_ZIP_PRINCIPAL

In [ ]:
FAMILIAS = {
    'Empresas': 'empresas',
    'Estabelecimentos': 'estabelecimentos',
    'Socios': 'socios',
    'Simples': 'simples',
    'Cnaes': 'dominios',
    'Motivos': 'dominios',
    'Municipios': 'dominios',
    'Naturezas': 'dominios',
    'Paises': 'dominios',
    'Qualificacoes': 'dominios',
}

def classificar_zip(nome: str) -> str:
    for prefixo, familia in FAMILIAS.items():
        if nome.startswith(prefixo):
            return familia
    raise ValueError(f'ZIP interno não reconhecido: {nome}')

def preparar_snapshot(caminho_zip_principal: Path, raiz_dados: Path, snapshot_mes: str) -> Path:
    if not caminho_zip_principal.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho_zip_principal}')

    snapshot = raiz_dados / snapshot_mes
    pacote_original = snapshot / '00_pacote_original'
    extraido_texto = snapshot / '02_extraido_texto'
    processado = snapshot / '03_processado'
    metadados = snapshot / '04_metadados'
    temporario = snapshot / '_tmp_subarquivos_zip'

    for pasta in [pacote_original, extraido_texto, processado / 'bronze', processado / 'silver', processado / 'gold', metadados, temporario]:
        pasta.mkdir(parents=True, exist_ok=True)

    destino_zip = pacote_original / caminho_zip_principal.name
    if caminho_zip_principal.resolve() != destino_zip.resolve():
        shutil.move(str(caminho_zip_principal), str(destino_zip))
    caminho_zip_principal = destino_zip

    with zipfile.ZipFile(caminho_zip_principal) as zip_principal:
        zip_principal.extractall(temporario)

    for zip_interno in sorted(temporario.glob('*.zip')):
        familia = classificar_zip(zip_interno.name)
        destino_familia = extraido_texto / familia
        destino_familia.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_interno) as zip_familia:
            zip_familia.extractall(destino_familia)

    shutil.rmtree(temporario)
    return snapshot

SNAPSHOT = preparar_snapshot(CAMINHO_ZIP_PRINCIPAL, RAIZ_DADOS, SNAPSHOT_MES)
SNAPSHOT

In [ ]:
partes = [
    '00_pacote_original',
    '02_extraido_texto',
    '03_processado',
    '04_metadados',
]

estrutura = {parte: (SNAPSHOT / parte).exists() for parte in partes}
estrutura

In [ ]:
familias = {
    'empresas': SNAPSHOT / '02_extraido_texto' / 'empresas',
    'estabelecimentos': SNAPSHOT / '02_extraido_texto' / 'estabelecimentos',
    'simples': SNAPSHOT / '02_extraido_texto' / 'simples',
    'socios': SNAPSHOT / '02_extraido_texto' / 'socios',
    'dominios': SNAPSHOT / '02_extraido_texto' / 'dominios',
}

resumo_familias = {}
for familia, pasta in familias.items():
    arquivos = sorted([arquivo for arquivo in pasta.iterdir() if arquivo.is_file()]) if pasta.exists() else []
    resumo_familias[familia] = {
        'quantidade_arquivos': len(arquivos),
        'primeiros_arquivos': [arquivo.name for arquivo in arquivos[:3]],
        'bytes_total': sum(arquivo.stat().st_size for arquivo in arquivos),
    }

resumo_familias

In [ ]:
def amostrar_linhas(caminho: Path, quantidade: int = 3):
    linhas = []
    with caminho.open('r', encoding='latin-1') as arquivo:
        for _ in range(quantidade):
            linha = arquivo.readline()
            if not linha:
                break
            linhas.append(linha.strip())
    return linhas

arquivos_amostra = {
    'cnae': SNAPSHOT / '02_extraido_texto' / 'dominios' / 'F.K03200$Z.D60411.CNAECSV',
    'municipios': SNAPSHOT / '02_extraido_texto' / 'dominios' / 'F.K03200$Z.D60411.MUNICCSV',
    'empresas': next((SNAPSHOT / '02_extraido_texto' / 'empresas').iterdir()),
    'estabelecimentos': next((SNAPSHOT / '02_extraido_texto' / 'estabelecimentos').iterdir()),
    'simples': next((SNAPSHOT / '02_extraido_texto' / 'simples').iterdir()),
}

{chave: amostrar_linhas(caminho) for chave, caminho in arquivos_amostra.items()}

In [ ]:
manifesto = {
    'snapshot_mes': SNAPSHOT_MES,
    'origem_zip': str((SNAPSHOT / '00_pacote_original' / '2026-04.zip')),
    'camada_intermediaria_persistida': False,
    'estrutura': estrutura,
    'resumo_familias': resumo_familias,
}

saida = SNAPSHOT / '04_metadados' / 'manifest_colab.json'
saida.write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8')
saida